# 30. 长上下文微调机制 | Long-Context Fine-Tuning Mechanics
**难度：** Medium | **环境：** CPU-first | **标签：** `后训练`, `长上下文`, `数据组织` | **目标人群：** 训练机制学习者

---

## 本节导读

长上下文微调不是把 `max_length` 调大。训练窗口必须同时容纳提示词、监督回答和模板 Token；长度分布又会改变 packing 效率、每步 Token 数、显存与训练时间。本节从样本长度出发，依次建立窗口预算、packing 账本和长度课程，再用质量与成本门槛筛选候选。

**关键词：** `length distribution`, `packing`, `context budget`, `length curriculum`

---

## 前置阅读

**导语：** 进入本节前，先理解 SFT 的监督边界与训练激活成本，再比较样本长度分布与目标窗口是否匹配。

- [13. SFT Training Loop | SFT 训练循环](./13_SFT_Training_Loop.ipynb)
- [19. Activation Checkpointing | 激活检查点](./19_Activation_Checkpointing.ipynb)

---


### Step 1：长度分布决定问题规模

先用目标分词器分别统计提示、回答、模板与完整样本的 Token 长度。长度分布、单样本准入和 Packing 是三个不同问题：P50 / P95 描述主体与长尾，窗口准入检查一条样本能否完整保留监督回答，Packing 只提高已经通过准入的短样本利用率。

| 观察项 | 回答的问题 | 对训练设计的影响 |
|:---|:---|:---|
| P50 / P95 | 典型样本与长尾样本有多长 | 选择基础窗口和长样本分支 |
| 溢出比例 | 多少样本无法完整进入目标窗口 | 截断、切块或扩大窗口 |
| 回答 Token | 监督区域需要预留多少空间 | 防止提示词挤占 labels |
| 长度分桶 | 各长度区间样本占比 | packing 与 curriculum 配比 |

![长上下文微调机制链](../docs/public/02_PyTorch_Algorithms/30_long_context_flow.svg)

### Step 2：窗口预算与 Packing 必须一起计算

单样本先按 `模板 + 提示 + 回答 + 终止符` 核对真实总量；只检查提示长度会把回答截断隐藏成可用样本。Packing 将多条已准入的短样本装入同一序列以减少填充，但必须同时保留样本边界、位置语义和标签掩码，不能让前一条回答继续监督后一条样本。

| 账本 | 关键不变量 | 失败表现 |
|:---|:---|:---|
| 单样本预算 | `prompt + response + template <= context` | 回答被截断或监督 Token 消失 |
| packed batch | 每个 bin 的 Token 总数不超过窗口 | OOM、越界或 padding 收益失真 |
| 样本边界 | 每条样本保留自己的监督区与终止符 | 跨样本学习和 loss 污染 |

![长上下文微调预算](../docs/public/02_PyTorch_Algorithms/30_long_context_budget.svg)

### Step 3：长度课程把覆盖率收益与训练代价对齐

训练可以先用短窗口建立稳定更新，再逐步提高长样本比例和目标窗口。阶段比较要固定每个优化器步的有效 Token 口径，并记录长度分桶采样比例；否则窗口变长带来的 Token 工作量变化会被误判为课程收益。候选只有在长样本覆盖率或独立长上下文评测改善，同时训练稳定性、峰值显存与每 Token 成本仍可接受时才进入下一阶段。

| 阶段 | 数据重点 | 需要比较的证据 |
|:---|:---|:---|
| 基线 | 主体短样本 | loss、tokens/s、峰值显存 |
| 过渡 | 增加中长样本比例 | packing 利用率、长样本质量、稳定性 |
| 目标 | 达到目标窗口 | 长上下文评测、每 Token 成本、失败/OOM |

### Step 4：实现监督准入、Packing 与候选决策

本题使用提示 / 回答 Token、Packing 容量和候选指标复现长上下文训练决策。骨架提供长度统计、输入校验、循环与报告装配；你只需补全单样本监督完整性准入、Packing 容器选择和候选硬门槛。测试覆盖窗口边界、样本与容量守恒、非法输入以及三类决策。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|:---|:---|:---|:---|
| TODO 1：判定监督完整性<br>`audit_context_fit` | 用提示、回答和模板总量判断样本能否完整进入窗口 | 只有完整保留回答的样本才能进入 fit 集合 | `test_context_fit_preserves_supervision`、`test_invalid_length_inputs` |
| TODO 2：选择 Packing 容器<br>`pack_sample_lengths` | 用 first-fit decreasing 为当前样本选择首个可容纳容器 | 每个样本只出现一次，任何容器不超容量 | `test_packing_preserves_capacity_and_samples`、`test_invalid_length_inputs` |
| TODO 3：执行候选硬门槛<br>`decide_long_context_candidate` | 区分质量失败、资源待调和全部通过 | 质量退化必须拒绝，资源超限不能被覆盖率收益抵消 | `test_candidate_decision_separates_quality_and_cost` |

## 测试

运行下方 CPU 机制测试，检查长度分布、窗口预算、Packing 容量和候选决策。


In [ ]:
from typing import Dict, List


In [ ]:
def summarize_length_distribution(
    lengths: List[int], short_threshold: int, long_threshold: int
) -> Dict[str, object]:
    """统计长度分桶、P50 与 P95。"""
    if short_threshold < 0 or long_threshold < short_threshold or any(length < 0 for length in lengths):
        raise ValueError("长度必须非负，且 short_threshold <= long_threshold")
    ordered = sorted(int(length) for length in lengths)
    buckets = {"short": sum(length <= short_threshold for length in ordered), "medium": sum(short_threshold < length <= long_threshold for length in ordered), "long": sum(length > long_threshold for length in ordered)}
    p50_tokens = ordered[max(0, int(0.50 * len(ordered) + 0.999999) - 1)] if ordered else 0
    p95_tokens = ordered[max(0, int(0.95 * len(ordered) + 0.999999) - 1)] if ordered else 0
    return {"count": len(ordered), "buckets": buckets, "p50_tokens": p50_tokens, "p95_tokens": p95_tokens}


def audit_context_fit(
    samples: List[Dict[str, int]],
    target_context_len: int,
    template_tokens: int,
) -> Dict[str, object]:
    """按真实提示与回答长度审计监督完整性。"""
    if target_context_len <= 0 or template_tokens < 0:
        raise ValueError("窗口必须为正，模板长度必须非负")
    records = []
    for sample in samples:
        prompt_tokens = int(sample["prompt_tokens"])
        response_tokens = int(sample["response_tokens"])
        if prompt_tokens < 0 or response_tokens <= 0:
            raise ValueError("提示长度必须非负，回答长度必须为正")
        # ==========================================
        # TODO 1：判定监督完整性
        # ==========================================
        # total_tokens = ???  # 当前样本包含模板、提示和回答的完整 Token 总量。
        # fits_context = ???  # 布尔值；仅当完整样本不超过目标窗口时为 True。
        raise NotImplementedError("TODO 1：判定监督完整性")
        records.append({"name": sample["name"], "total_tokens": total_tokens, "supervised_tokens": response_tokens if fits_context else 0, "decision": "fit" if fits_context else "overflow"})
    fit_names = [item["name"] for item in records if item["decision"] == "fit"]
    overflow_names = [item["name"] for item in records if item["decision"] == "overflow"]
    return {"records": records, "fit_names": fit_names, "overflow_names": overflow_names, "fit_rate": len(fit_names) / len(records) if records else 0.0}


def pack_sample_lengths(lengths: List[int], max_tokens_per_bin: int) -> Dict[str, object]:
    """使用 first-fit decreasing 构造 Packing 容器。"""
    if max_tokens_per_bin <= 0 or any(length <= 0 or length > max_tokens_per_bin for length in lengths):
        raise ValueError("每个长度必须位于 (0, max_tokens_per_bin] 区间")
    bins: List[List[int]] = []
    for length in sorted(lengths, reverse=True):
        # ==========================================
        # TODO 2：选择 Packing 容器
        # ==========================================
        # target_bin = ???  # 第一个还能容纳当前样本的容器；不存在时为 None。
        raise NotImplementedError("TODO 2：选择 Packing 容器")
        if target_bin is None:
            bins.append([length])
        else:
            target_bin.append(length)
    capacity = len(bins) * max_tokens_per_bin
    return {"bins": bins, "utilization": sum(lengths) / capacity if capacity else 0.0}


def decide_long_context_candidate(
    baseline: Dict[str, float],
    candidate: Dict[str, float],
    max_memory_ratio: float,
    max_time_per_token_ratio: float,
) -> Dict[str, object]:
    """联合覆盖率、质量与资源门槛形成决策。"""
    if baseline["peak_memory_gb"] <= 0 or baseline["time_per_token_ms"] <= 0:
        raise ValueError("基线资源指标必须为正数")
    coverage_gain = candidate["coverage"] - baseline["coverage"]
    quality_delta = candidate["quality"] - baseline["quality"]
    memory_ratio = candidate["peak_memory_gb"] / baseline["peak_memory_gb"]
    time_per_token_ratio = candidate["time_per_token_ms"] / baseline["time_per_token_ms"]
    # ==========================================
    # TODO 3：执行候选硬门槛
    # ==========================================
    # hard_failure = ???  # 布尔值；核心质量相对基线退化时为 True。
    # needs_tuning = ???  # 布尔值；覆盖无增益或任一资源倍率越界时为 True。
    # decision = ???  # 质量失败、需要调参与全部通过分别映射到 reject / tune / accept。
    raise NotImplementedError("TODO 3：执行候选硬门槛")
    return {"coverage_gain": coverage_gain, "quality_delta": quality_delta, "memory_ratio": memory_ratio, "time_per_token_ratio": time_per_token_ratio, "decision": decision}

In [ ]:
def _assert_raises_value_error(callback) -> None:
    """确认非法长度、窗口或容量被显式拒绝。"""
    try:
        callback()
    except ValueError:
        return
    raise AssertionError("非法输入应抛出 ValueError")


def test_context_fit_preserves_supervision() -> None:
    """长度统计与窗口准入必须保留真实回答监督。"""
    summary = summarize_length_distribution([64, 128, 300, 900], 128, 512)
    assert summary["buckets"] == {"short": 2, "medium": 1, "long": 1}
    assert summary["p50_tokens"] == 128
    assert summary["p95_tokens"] == 900
    samples = [
        {"name": "short", "prompt_tokens": 300, "response_tokens": 80},
        {"name": "boundary", "prompt_tokens": 1760, "response_tokens": 256},
        {"name": "overflow", "prompt_tokens": 1761, "response_tokens": 256},
    ]
    plan = audit_context_fit(samples, 2048, 32)
    assert plan["fit_names"] == ["short", "boundary"]
    assert plan["overflow_names"] == ["overflow"]
    assert plan["records"][1]["total_tokens"] == 2048
    assert plan["records"][2]["supervised_tokens"] == 0


def test_packing_preserves_capacity_and_samples() -> None:
    """Packing 不得超出 bin 容量，也不能丢失样本。"""
    report = pack_sample_lengths([7, 6, 4, 3], 10)
    assert report["bins"] == [[7, 3], [6, 4]]
    assert all(sum(bin_lengths) <= 10 for bin_lengths in report["bins"])
    assert sorted(sum(report["bins"], [])) == [3, 4, 6, 7]
    assert report["utilization"] == 1.0


def test_invalid_length_inputs() -> None:
    """负长度、空监督和超长 Packing 样本必须被拒绝。"""
    _assert_raises_value_error(lambda: summarize_length_distribution([-1, 8], 16, 32))
    _assert_raises_value_error(lambda: audit_context_fit([{"name": "bad", "prompt_tokens": 64, "response_tokens": 0}], 128, 8))
    _assert_raises_value_error(lambda: pack_sample_lengths([129], 128))


def test_candidate_decision_separates_quality_and_cost() -> None:
    """质量硬门槛与资源软门槛必须产生不同决策。"""
    baseline = {"coverage": 0.45, "quality": 0.61, "peak_memory_gb": 14.0, "time_per_token_ms": 2.0}
    accepted = {"coverage": 0.80, "quality": 0.64, "peak_memory_gb": 18.0, "time_per_token_ms": 2.4}
    tuned = {**accepted, "peak_memory_gb": 21.0}
    rejected = {**accepted, "quality": 0.60}
    assert decide_long_context_candidate(baseline, accepted, 1.4, 1.3)["decision"] == "accept"
    assert decide_long_context_candidate(baseline, tuned, 1.4, 1.3)["decision"] == "tune"
    assert decide_long_context_candidate(baseline, rejected, 1.4, 1.3)["decision"] == "reject"


try:
    test_context_fit_preserves_supervision()
    test_packing_preserves_capacity_and_samples()
    test_invalid_length_inputs()
    test_candidate_decision_separates_quality_and_cost()
    print("✅ 长度、窗口、packing 与候选决策测试通过。")
except (AttributeError, NameError, TypeError, ValueError, AssertionError) as error:
    raise NotImplementedError("请先完成 TODO 代码！") from error

---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
def summarize_length_distribution(
    lengths: List[int], short_threshold: int, long_threshold: int
) -> Dict[str, object]:
    """统计长度分桶、P50 与 P95。"""
    if short_threshold < 0 or long_threshold < short_threshold or any(length < 0 for length in lengths):
        raise ValueError("长度必须非负，且 short_threshold <= long_threshold")
    ordered = sorted(int(length) for length in lengths)
    buckets = {"short": sum(length <= short_threshold for length in ordered), "medium": sum(short_threshold < length <= long_threshold for length in ordered), "long": sum(length > long_threshold for length in ordered)}
    p50_tokens = ordered[max(0, int(0.50 * len(ordered) + 0.999999) - 1)] if ordered else 0
    p95_tokens = ordered[max(0, int(0.95 * len(ordered) + 0.999999) - 1)] if ordered else 0
    return {
        "count": len(ordered),
        "buckets": buckets,
        "p50_tokens": p50_tokens,
        "p95_tokens": p95_tokens,
    }


def audit_context_fit(
    samples: List[Dict[str, int]],
    target_context_len: int,
    template_tokens: int,
) -> Dict[str, object]:
    """按真实提示与回答长度审计监督完整性。"""
    if target_context_len <= 0 or template_tokens < 0:
        raise ValueError("窗口必须为正，模板长度必须非负")
    records = []
    for sample in samples:
        prompt_tokens = int(sample["prompt_tokens"])
        response_tokens = int(sample["response_tokens"])
        if prompt_tokens < 0 or response_tokens <= 0:
            raise ValueError("提示长度必须非负，回答长度必须为正")
        # TODO 1：判定监督完整性。
        total_tokens = template_tokens + prompt_tokens + response_tokens
        fits_context = total_tokens <= target_context_len
        records.append({"name": sample["name"], "total_tokens": total_tokens, "supervised_tokens": response_tokens if fits_context else 0, "decision": "fit" if fits_context else "overflow"})
    fit_names = [item["name"] for item in records if item["decision"] == "fit"]
    overflow_names = [item["name"] for item in records if item["decision"] == "overflow"]
    return {"records": records, "fit_names": fit_names, "overflow_names": overflow_names, "fit_rate": len(fit_names) / len(records) if records else 0.0}


def pack_sample_lengths(lengths: List[int], max_tokens_per_bin: int) -> Dict[str, object]:
    """使用 first-fit decreasing 构造 packing bins。"""
    if max_tokens_per_bin <= 0 or any(length <= 0 or length > max_tokens_per_bin for length in lengths):
        raise ValueError("每个长度必须位于 (0, max_tokens_per_bin] 区间")
    # TODO 2：选择 Packing 容器。
    bins: List[List[int]] = []
    for length in sorted(lengths, reverse=True):
        target_bin = next((bin_lengths for bin_lengths in bins if sum(bin_lengths) + length <= max_tokens_per_bin), None)
        if target_bin is None:
            bins.append([length])
        else:
            target_bin.append(length)
    capacity = len(bins) * max_tokens_per_bin
    return {
        "bins": bins,
        "utilization": sum(lengths) / capacity if capacity else 0.0,
    }


def decide_long_context_candidate(
    baseline: Dict[str, float],
    candidate: Dict[str, float],
    max_memory_ratio: float,
    max_time_per_token_ratio: float,
) -> Dict[str, object]:
    """联合覆盖率、质量与资源门槛形成决策。"""
    if baseline["peak_memory_gb"] <= 0 or baseline["time_per_token_ms"] <= 0:
        raise ValueError("基线显存与单 Token 时间必须为正数")
    coverage_gain = candidate["coverage"] - baseline["coverage"]
    quality_delta = candidate["quality"] - baseline["quality"]
    memory_ratio = candidate["peak_memory_gb"] / baseline["peak_memory_gb"]
    time_per_token_ratio = candidate["time_per_token_ms"] / baseline["time_per_token_ms"]
    # TODO 3：执行候选硬门槛。
    hard_failure = quality_delta < 0
    needs_tuning = coverage_gain <= 0 or memory_ratio > max_memory_ratio or time_per_token_ratio > max_time_per_token_ratio
    if hard_failure:
        decision = "reject"
    elif needs_tuning:
        decision = "tune"
    else:
        decision = "accept"
    return {
        "coverage_gain": coverage_gain,
        "quality_delta": quality_delta,
        "memory_ratio": memory_ratio,
        "time_per_token_ratio": time_per_token_ratio,
        "decision": decision,
    }

### 解析

#### TODO 1：判定监督完整性

- 单样本准入必须计算模板、提示与真实回答的完整 Token 总量；只看提示会掩盖监督回答被截断。
- 答案仅让完整样本进入 fit 集合，并由边界样本和空监督反例验证准入条件。

#### TODO 2：选择 Packing 容器

- 答案按长度降序遍历，并把当前样本放入第一个仍有容量的容器；不存在时才新建容器。
- 这一实现对应 First-fit decreasing；测试检查每个容器不超容量、样本总量守恒以及超长样本被拒绝。

#### TODO 3：执行候选硬门槛

- 质量退化属于不可被覆盖率收益抵消的硬失败；覆盖率无增益或资源越界属于可继续调整的候选。
- 答案按 reject、tune、accept 的优先级执行门槛，测试分别构造三类结果。

### Step 5：可选 GPU 实验——序列长度与训练资源曲线

CPU 题目确认长度分布、窗口预算与 Packing 决策；GPU 实验先固定真实模型、batch 和输入语义扫描序列长度，再固定原始样本与有效监督 Token 比较 padding 和边界隔离 Packing。

| 学习问题 | CPU 已确认 | GPU 要确认与后续使用 |
| --- | --- | --- |
| 序列变长如何改变训练资源 | 窗口、覆盖率与候选 gate | 真实模型反向传播的时间、每 Token 时间和显存曲线 |
| Packing 是否在保持样本边界后减少无效计算 | Padding 与 Packing 的 Token 账本 | 固定原始样本和有效监督 Token，比较处理 Token、吞吐、显存及边界隔离 |

#### 5.1 环境与固定工作负载

本实验使用 [`PT-SFT-CUDA`](../docs/gpu_environment_assets.md#pt-sft-cuda)。首次运行先完成[后训练实验 Quickstart](../topic_discussion/post_training_optimization/quickstart.md#pt-sft-cuda-quickstart)；已有环境只需通过 Profile 预检，并从同一解释器启动 JupyterLab。页面、工作负载和结果状态见[后训练优化实验资产表](../topic_discussion/post_training_optimization/experiment_assets.md)。

| 配置层 | 本节固定口径 | 修改规则 |
| --- | --- | --- |
| 运行环境 | `PT-SFT-CUDA`；单张 NVIDIA GPU；参考显存 8 GiB | 预检失败时停止实验 |
| 模型 / 数据来源 | `Qwen/Qwen2.5-0.5B`；内置固定文本 | 自动固定模型 commit；本实验不下载数据集 |
| 工作负载 | 长度曲线：batch 1、长度 128 / 256 / 512；Packing：4 条固定样本 | 改变长度矩阵、样本长度或次数后递增 workload version |
| 配对变量 | 长度曲线只改变序列长度；Packing 对照只改变 padding / 边界隔离组织方式 | 模型、dtype、SDPA backend、原始样本、随机种子和硬件保持一致 |
| 测量方法 | loss 与梯度有限性、有效监督 Token、处理 Token、前向 / 反向时间、吞吐和峰值显存 | Packing 必须先通过标签边界与跨样本 Attention 隔离检查 |
| 结果资产 | `benchmarks/results/pt_long_context/pt-long-context_<run_label>.json` | 正式复测使用唯一标签并保留旧结果 |

In [1]:
RUN_MODE = "dry_run"  # 默认值 dry_run：只登记计划；改为 real_gpu 执行长度曲线与 Packing 对照。
RUN_LABEL = "default"  # 默认值 default；正式复测改为唯一标签，避免覆盖旧结果。
ENVIRONMENT_PROFILE = "PT-SFT-CUDA"  # 默认值 PT-SFT-CUDA；运行前必须通过同名 Profile 预检。
MODEL_ID = "Qwen/Qwen2.5-0.5B"  # 默认基础模型；更换后不得与旧结果直接合并。
MODEL_REVISION = "main"  # 默认 main；5.2 自动解析并记录实际模型 commit。
WORKLOAD_ID = "pt-long-context"  # 稳定工作负载身份。
WORKLOAD_VERSION = "v4"  # v4 为显存趋势检查增加分配器波动容差。
SEQUENCE_LENGTHS = [128, 256, 512]  # 默认三个 Token 长度点；增加长度会提高耗时与显存。
PACKING_SAMPLE_LENGTHS = [48, 96, 32, 64]  # 四条固定样本的 Token 长度；padding 与 Packing 复用同一组 Token。
BATCH_SIZE = 1  # 默认值 1；改变后同时改变 Token 工作量与峰值显存。
WARMUP_STEPS = 2  # 默认值 2 次；不进入计时统计。
REPEAT_STEPS = 10  # 默认值 10 次；用于降低单次 GPU 抖动对中位数和 P95 的影响。
MEMORY_TREND_TOLERANCE = 0.005  # 默认值 0.005；允许相邻长度点出现 0.5% 以内的分配器波动。
ATTENTION_BACKEND = "sdpa"  # 默认值 sdpa；改变 backend 后结果不可与本工作负载直接合并。
SEED = 42  # 默认值 42；固定模型和输入构造。
LOCAL_FILES_ONLY = False  # 默认值 False；已缓存相同 commit 且离线运行时改为 True。
OUTPUT_RELATIVE_PATH = f"benchmarks/results/pt_long_context/{WORKLOAD_ID}_{RUN_LABEL}.json"  # 由运行标签派生的仓库相对路径。

#### 5.2 执行并保存 JSON

5.2 先从同一模型快照构造三个长度点，再用同一组原始样本构造 padding 与边界隔离 Packing。两组实验分别计时；每次测量前清空梯度并重置峰值显存统计。

In [2]:
import gc
import platform
import time

import torch
from tools.notebook_runtime import failure_record, package_version, resolve_model_commit, save_json
from tools.project_runtime import resolve_project_root
from tools.training_mechanism_runtime import cuda_hardware_snapshot, gradient_l2_norm, median_and_p95

REPO_ROOT = resolve_project_root()
OUTPUT_PATH = REPO_ROOT / OUTPUT_RELATIVE_PATH
result = {
    "schema_version": "post-training-mechanism-gpu/v1", "semantic_id": "PT-LONG-CONTEXT",
    "environment_profile": ENVIRONMENT_PROFILE, "run_mode": RUN_MODE, "run_label": RUN_LABEL,
    "framework": {"python": platform.python_version(), "torch": torch.__version__, "transformers": package_version("transformers")},
    "hardware": {"cuda_available": torch.cuda.is_available(), "cuda": torch.version.cuda},
    "model": {"id": MODEL_ID, "requested_revision": MODEL_REVISION, "resolved_commit": None},
    "workload": {"id": WORKLOAD_ID, "version": WORKLOAD_VERSION, "sequence_lengths": SEQUENCE_LENGTHS, "packing_sample_lengths": PACKING_SAMPLE_LENGTHS, "batch_size": BATCH_SIZE, "warmup_steps": WARMUP_STEPS, "repeat_steps": REPEAT_STEPS, "memory_trend_tolerance": MEMORY_TREND_TOLERANCE, "attention_backend": ATTENTION_BACKEND, "seed": SEED},
    "measurements": [], "packing_comparison": [], "failure": None, "evidence_level": "dry_run", "decision": "not_measured",
}
execution_stage = "environment_preflight"
if RUN_MODE == "real_gpu":
    model = None
    try:
        if not torch.cuda.is_available():
            raise RuntimeError("RUN_MODE=real_gpu，但 CUDA 不可用")
        if not SEQUENCE_LENGTHS or sorted(set(SEQUENCE_LENGTHS)) != SEQUENCE_LENGTHS or min(SEQUENCE_LENGTHS) <= 1:
            raise ValueError("SEQUENCE_LENGTHS 必须是严格递增且大于 1 的唯一长度列表")
        if len(PACKING_SAMPLE_LENGTHS) < 2 or min(PACKING_SAMPLE_LENGTHS) <= 1:
            raise ValueError("PACKING_SAMPLE_LENGTHS 至少包含两个大于 1 的样本长度")
        from transformers import AutoModelForCausalLM, AutoTokenizer
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        device = torch.device("cuda")
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        execution_stage = "revision_and_model"
        model_commit, revision_source = resolve_model_commit(MODEL_ID, MODEL_REVISION, local_files_only=LOCAL_FILES_ONLY)
        result["model"].update({"resolved_commit": model_commit, "resolution_source": revision_source})
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_commit, local_files_only=LOCAL_FILES_ONLY)
        seed_ids = tokenizer("Long context training must preserve supervision while controlling memory and time.", add_special_tokens=False)["input_ids"]
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=model_commit, local_files_only=LOCAL_FILES_ONLY, dtype=dtype, attn_implementation=ATTENTION_BACKEND).to(device)
        model.config.use_cache = False
        result["workload"]["parameter_dtype"] = str(dtype).replace("torch.", "")
        model.train()
        execution_stage = "length_matrix"
        for sequence_length in SEQUENCE_LENGTHS:
            repeated = (seed_ids * ((sequence_length + len(seed_ids) - 1) // len(seed_ids)))[:sequence_length]
            input_ids = torch.tensor([repeated] * BATCH_SIZE, dtype=torch.long, device=device)
            batch = {"input_ids": input_ids, "attention_mask": torch.ones_like(input_ids), "labels": input_ids.clone()}
            timings, losses, gradient_norms = [], [], []
            for step in range(WARMUP_STEPS + REPEAT_STEPS):
                model.zero_grad(set_to_none=True)
                if step == WARMUP_STEPS:
                    torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
                started = time.perf_counter()
                with torch.autocast("cuda", dtype=dtype):
                    loss = model(**batch).loss
                loss.backward()
                torch.cuda.synchronize()
                if step >= WARMUP_STEPS:
                    timings.append((time.perf_counter() - started) * 1000)
                    losses.append(float(loss.detach().item()))
                    gradient_norms.append(gradient_l2_norm(model.parameters()))
            median_ms, p95_ms = median_and_p95(timings)
            result["measurements"].append({"sequence_length": sequence_length, "tokens_per_step": sequence_length * BATCH_SIZE, "median_step_ms": median_ms, "p95_step_ms": p95_ms, "median_ms_per_token": median_ms / (sequence_length * BATCH_SIZE), "losses": losses, "gradient_norms": gradient_norms, "peak_allocated_bytes": int(torch.cuda.max_memory_allocated()), "peak_reserved_bytes": int(torch.cuda.max_memory_reserved())})

        execution_stage = "packing_comparison"
        packing_texts = [
            "Explain why sample boundaries matter in supervised fine-tuning.",
            "Describe how padding changes the amount of work in a training batch.",
            "Give one invariant that a packed sequence must preserve.",
            "Summarize the difference between useful tokens and processed tokens.",
        ]
        sample_token_ids = []
        for text, target_length in zip(packing_texts, PACKING_SAMPLE_LENGTHS):
            base_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            expanded = (base_ids * ((target_length + len(base_ids) - 1) // len(base_ids)))[:target_length]
            sample_token_ids.append(expanded)

        pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        padded_length = max(PACKING_SAMPLE_LENGTHS)
        padded_input_ids = torch.full((len(sample_token_ids), padded_length), pad_token_id, dtype=torch.long, device=device)
        padded_attention_mask = torch.zeros_like(padded_input_ids)
        padded_labels = torch.full_like(padded_input_ids, -100)
        for row, token_ids in enumerate(sample_token_ids):
            length = len(token_ids)
            padded_input_ids[row, :length] = torch.tensor(token_ids, dtype=torch.long, device=device)
            padded_attention_mask[row, :length] = 1
            padded_labels[row, :length] = padded_input_ids[row, :length]
            padded_labels[row, 0] = -100
        padded_batch = {"input_ids": padded_input_ids, "attention_mask": padded_attention_mask, "labels": padded_labels}

        packed_ids = torch.tensor([sum(sample_token_ids, [])], dtype=torch.long, device=device)
        total_packed_length = packed_ids.shape[1]
        packed_labels = packed_ids.clone()
        packed_position_ids = torch.empty_like(packed_ids)
        segment_ids = torch.empty(total_packed_length, dtype=torch.long, device=device)
        packed_attention_mask = torch.full((1, 1, total_packed_length, total_packed_length), torch.finfo(dtype).min, dtype=dtype, device=device)
        segment_starts, offset = [], 0
        for segment_id, token_ids in enumerate(sample_token_ids):
            segment_starts.append(offset)
            end = offset + len(token_ids)
            packed_labels[0, offset] = -100
            packed_position_ids[0, offset:end] = torch.arange(len(token_ids), device=device)
            segment_ids[offset:end] = segment_id
            local_causal = torch.tril(torch.ones((len(token_ids), len(token_ids)), dtype=torch.bool, device=device))
            packed_attention_mask[0, 0, offset:end, offset:end].masked_fill_(local_causal, 0.0)
            offset = end
        packed_batch = {"input_ids": packed_ids, "attention_mask": packed_attention_mask, "position_ids": packed_position_ids, "labels": packed_labels}

        def measure_training_batch(candidate_name, candidate_batch):
            timings, losses, gradient_norms = [], [], []
            for step in range(WARMUP_STEPS + REPEAT_STEPS):
                model.zero_grad(set_to_none=True)
                if step == WARMUP_STEPS:
                    torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
                started = time.perf_counter()
                with torch.autocast("cuda", dtype=dtype):
                    loss = model(**candidate_batch).loss
                loss.backward()
                torch.cuda.synchronize()
                if step >= WARMUP_STEPS:
                    timings.append((time.perf_counter() - started) * 1000)
                    losses.append(float(loss.detach().item()))
                    gradient_norms.append(gradient_l2_norm(model.parameters()))
            median_ms, p95_ms = median_and_p95(timings)
            effective_tokens = int(candidate_batch["labels"][:, 1:].ne(-100).sum().item())
            processed_tokens = int(candidate_batch["input_ids"].numel())
            return {"candidate": candidate_name, "processed_tokens": processed_tokens, "effective_supervised_tokens": effective_tokens, "padding_tokens": processed_tokens - sum(PACKING_SAMPLE_LENGTHS), "padding_ratio": 1.0 - sum(PACKING_SAMPLE_LENGTHS) / processed_tokens, "median_step_ms": median_ms, "p95_step_ms": p95_ms, "effective_tokens_per_second": effective_tokens / (median_ms / 1000.0), "losses": losses, "gradient_norms": gradient_norms, "peak_allocated_bytes": int(torch.cuda.max_memory_allocated()), "peak_reserved_bytes": int(torch.cuda.max_memory_reserved())}

        result["packing_comparison"].append(measure_training_batch("padding", padded_batch))
        result["packing_comparison"].append(measure_training_batch("boundary_isolated_packing", packed_batch))
        cross_segment = segment_ids[:, None].ne(segment_ids[None, :])
        packing_checks = {
            "same_effective_supervised_tokens": result["packing_comparison"][0]["effective_supervised_tokens"] == result["packing_comparison"][1]["effective_supervised_tokens"],
            "segment_start_labels_masked": bool(packed_labels[0, segment_starts].eq(-100).all().item()),
            "position_ids_reset": bool(packed_position_ids[0, segment_starts].eq(0).all().item()),
            "cross_sample_attention_blocked": bool((packed_attention_mask[0, 0][cross_segment] < 0).all().item()),
        }
        result["packing_checks"] = packing_checks
        finite = all(torch.isfinite(torch.tensor(item["losses"] + item["gradient_norms"])).all().item() for item in result["measurements"])
        packing_finite = all(torch.isfinite(torch.tensor(item["losses"] + item["gradient_norms"])).all().item() for item in result["packing_comparison"])
        memory_trend_passed = all(right["peak_allocated_bytes"] >= left["peak_allocated_bytes"] * (1 - MEMORY_TREND_TOLERANCE) for left, right in zip(result["measurements"], result["measurements"][1:]))
        result["length_checks"] = {"finite_losses_and_gradients": finite, "memory_non_decreasing_with_tolerance": memory_trend_passed, "memory_trend_tolerance": MEMORY_TREND_TOLERANCE}
        passed = finite and packing_finite and memory_trend_passed and all(packing_checks.values())
        result.update({"hardware": cuda_hardware_snapshot(), "evidence_level": "real_model_length_and_packing_mechanism_benchmark", "decision": "accept_mechanism" if passed else "tune"})
    except Exception as error:
        result["failure"] = failure_record(execution_stage, error)
        result["evidence_level"] = "failed_real_gpu_attempt"
        result["decision"] = "reject_until_fixed"
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
elif RUN_MODE == "dry_run":
    result["decision"] = "ready_to_measure"
else:
    raise ValueError("RUN_MODE 只能是 dry_run 或 real_gpu")
save_json(OUTPUT_PATH, result)
print(OUTPUT_PATH)
print(result["decision"], result["failure"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/data/llm-algo-leetcode-datawhale/llm-algo-leetcode/benchmarks/results/pt_long_context/pt-long-context_rtx5070ti_length_packing_v4.json
accept_mechanism None


#### 5.3 读取长度曲线与 Packing 对照

先检查失败阶段和模型 commit，再读取长度曲线以及 padding / 边界隔离 Packing 的有效监督 Token、处理 Token、边界检查、吞吐和峰值显存。

In [3]:
import json
if not OUTPUT_PATH.exists():
    print(f"尚未找到结果文件：{OUTPUT_PATH}")
else:
    record = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
    print(json.dumps({key: record.get(key) for key in ("run_label", "model", "failure", "evidence_level", "decision")}, ensure_ascii=False, indent=2))
    for item in record.get("measurements", []):
        print({key: item.get(key) for key in ("sequence_length", "tokens_per_step", "median_step_ms", "p95_step_ms", "median_ms_per_token", "peak_allocated_bytes", "peak_reserved_bytes")})
    print("长度趋势检查", record.get("length_checks"))
    print("Packing 边界检查", record.get("packing_checks"))
    for item in record.get("packing_comparison", []):
        print({key: item.get(key) for key in ("candidate", "processed_tokens", "effective_supervised_tokens", "padding_ratio", "median_step_ms", "p95_step_ms", "effective_tokens_per_second", "peak_allocated_bytes")})

{
  "run_label": "rtx5070ti_length_packing_v4",
  "model": {
    "id": "Qwen/Qwen2.5-0.5B",
    "requested_revision": "main",
    "resolved_commit": "060db6499f32faf8b98477b0a26969ef7d8b9987",
    "resolution_source": "huggingface_hub_api"
  },
  "failure": null,
  "evidence_level": "real_model_length_and_packing_mechanism_benchmark",
  "decision": "accept_mechanism"
}
{'sequence_length': 128, 'tokens_per_step': 128, 'median_step_ms': 29.162819497287273, 'p95_step_ms': 29.512763998354785, 'median_ms_per_token': 0.22783452732255682, 'peak_allocated_bytes': 3103056384, 'peak_reserved_bytes': 3791650816}
{'sequence_length': 256, 'tokens_per_step': 256, 'median_step_ms': 47.058541007572785, 'p95_step_ms': 88.00304199394304, 'median_ms_per_token': 0.1838224258108312, 'peak_allocated_bytes': 3106926080, 'peak_reserved_bytes': 4410310656}
{'sequence_length': 512, 'tokens_per_step': 512, 'median_step_ms': 79.51640000101179, 'p95_step_ms': 119.30764499993529, 'median_ms_per_token': 0.1553054687

#### 5.4 解释长度、Packing 与资源证据

先检查 `failure`、模型 commit 和两组实验口径。长度曲线用于判断资源随长度的变化；Packing 只有在有效监督 Token 一致、样本起点标签被屏蔽、位置编号重置且跨样本 Attention 被阻断后，才能解释吞吐和显存差异。RTX 5070 Ti 的 `rtx5070ti_length_packing_v4` 实测已通过长度趋势与四项 Packing 边界检查。

| 观察项 | 读取字段 | 当前结果 | 解释与动作 |
| --- | --- | --- | --- |
| 运行与口径 | `failure`、model commit、sequence lengths、packing sample lengths | 无失败；长度 128 / 256 / 512，固定四条 Packing 样本 | 当前矩阵可复查 |
| 长度机制 | `tokens_per_step`、`gradient_norms`、`length_checks` | 三个长度点均有有限梯度；0.5% 容差下显存趋势通过 | 256→512 的微小分配波动不再误判为机制失败 |
| Packing 正确性 | `packing_checks`、`effective_supervised_tokens` | 四项检查全部通过；两候选均为 236 个有效监督 Token | 性能差异来自样本组织，不是监督量变化 |
| 资源与性能 | 处理 Token、`median_step_ms`、有效 Token/s、峰值显存 | 384→240 Token；63.4→48.8 ms；3724→4835 Token/s；最高峰值分配约 2.90 GiB | 边界隔离 Packing 减少 37.5% 处理量，中位时间下降约 23%，有效吞吐提升约 30% |
| 证据与决策 | `evidence_level`、`decision` | `accept_mechanism` | 接受当前 Packing 机制证据，不把重复文本 loss 当作任务质量 |

## 相关阅读

完成长度分布、上下文预算和 CPU 观察后，可以继续阅读长上下文方法、显存动作与真实微调项目。

- [LongLoRA 原论文：Efficient Fine-tuning of Long-Context Large Language Models](https://arxiv.org/abs/2309.12307)
- [Hugging Face Transformers：长文本生成文档](https://huggingface.co/docs/transformers/main/en/llm_tutorial)
- [32. SFT Data Admission and Length Audit | SFT 数据准入与长度审计](./32_Data_Engineering_for_SFT.ipynb)
- [42. Activation Offload | 激活卸载](./42_Activation_Offload.ipynb)
- [62. Instruction Fine-Tuning Project | 指令微调项目](./62_Instruction_Fine_Tuning_Project.ipynb)
